# Isoglot 02: judge the baseline answers

Labels each of the 1,200 baseline answers **Jailbroken**, **Refused** or **Deflected**
with TukaBench's own judge prompt, using Claude Sonnet 5.5 as the judge, then builds
the per-language table. (TukaBench used GPT-4.1, so our rates are not one-to-one
comparable with the paper's.)

**Before running:**
1. Settings → Accelerator → **None** (no GPU needed). Settings → Internet **on**.
2. Add-ons → Secrets, both ticked for this notebook:
   - **`HF_TOKEN`**: your Hugging Face Write token (same as notebook 01)
   - **`ANTHROPIC_API_KEY`**: your Anthropic API key. Never paste it into a cell.
3. Run it in the background: **Save Version → Save & Run All (Commit)**.

**Spend:** the run stops itself once it has spent `CAP` dollars (below), and the
pilot stops it before the full run if the projected cost is over the cap.


In [ ]:
%%capture
!pip install -q "isoglot[judge] @ git+https://github.com/nftkingiii/isoglot"


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["ANTHROPIC_API_KEY"] = secrets.get_secret("ANTHROPIC_API_KEY")

from pathlib import Path
from huggingface_hub import HfApi, snapshot_download, whoami
api = HfApi()
REPO = f"{whoami()['name']}/isoglot-runs"
RUNS = Path("/kaggle/working/runs")
snapshot_download(REPO, repo_type="dataset", local_dir=RUNS, allow_patterns=["base/*/*.jsonl"])
print("answers downloaded:", len(list(RUNS.glob("base/*/*.jsonl"))), "files")


In [ ]:
from isoglot.judge import CLAUDE_PRICES, SpendMeter, TukaJudge, anthropic_client

JUDGE_MODEL = "claude-sonnet-5-5"
CAP = 5.00   # USD, hard stop for this notebook

meter = SpendMeter(CAP, *CLAUDE_PRICES[JUDGE_MODEL])
judge = TukaJudge(anthropic_client(JUDGE_MODEL, meter), JUDGE_MODEL)


## Pilot: 12 answers

One answer from every set and language, to check the labels look sensible and
to project the cost of the full run before spending it.

In [ ]:
from isoglot import LANGS, SETS
from isoglot.generate import load_answers
from isoglot.judge import judge_answers

pilot = [load_answers(RUNS / "base" / s / f"{l}.jsonl")[1] for s in SETS for l in LANGS]
for r in judge_answers(pilot, judge):
    print(f"{r['set']:17} {r['lang']} | {r['label']:10} | {r['reason'][:90]}")

per_answer = meter.spent_usd / meter.calls
projected = per_answer * 1200
print("fallbacks:", meter.fallbacks, "| judge declined:", meter.refusals)
print(f"pilot spent ${meter.spent_usd:.3f}; projected full run ${projected:.2f} (cap ${CAP:.2f})")
assert projected < CAP - meter.spent_usd, "projected cost is over the cap: stopping before the full run"


## Full run (resumable)

In [ ]:
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import EntryNotFoundError
from isoglot.judge import judge_chunks

def remote(path):
    return path.relative_to(RUNS).as_posix()

def fetch(path):
    try:
        hf_hub_download(REPO, remote(path), repo_type="dataset", local_dir=RUNS)
        print("restored", remote(path), "from the Hub")
    except EntryNotFoundError:
        pass

def publish(path):
    api.upload_file(path_or_fileobj=str(path), path_in_repo=remote(path),
                    repo_id=REPO, repo_type="dataset")

judged = judge_chunks(RUNS, judge, meter, sets=SETS, langs=LANGS, variant="base",
                      fetch=fetch, publish=publish)
publish(judged)
print(f"done: ${meter.spent_usd:.2f} over {meter.calls} calls "
      f"({meter.input_tokens:,} in / {meter.output_tokens:,} out tokens)")
print("fallbacks:", meter.fallbacks, "| judge declined:", meter.refusals)


## Baseline table

In [ ]:
import json
from isoglot.metrics import summarize, to_markdown

rows = [json.loads(l) for l in open(judged, encoding='utf-8')]
table = to_markdown(summarize(rows))
print(table)
header = (f'Judge: {JUDGE_MODEL} (TukaBench rubric). Fallbacks: {meter.fallbacks}, '
          f'declined: {meter.refusals}. Spend: ${meter.spent_usd:.2f}.')
report = RUNS / 'base' / 'report.md'
report.write_text(header + chr(10) * 2 + table, encoding='utf-8')
publish(report)
